# 🧠 Custom Training Loops in Keras


## 📋 Overview

I write my own training loop for a Keras model using `tf.GradientTape` instead of calling `model.fit()` — computing the forward pass, loss, and gradients manually, then applying the update step myself. I build this up in stages: a bare-bones loop, then one that tracks accuracy, then one that hooks in a custom callback for logging. Partway through, the material pivots to building an ordinary Functional API model trained with the standard `model.fit()`/`model.evaluate()` — I flag that shift clearly since it's a different topic bolted onto the same notebook. I close by working back through every step as a set of practice exercises.

Coming from a control-systems mindset, `model.fit()` is like using a chip vendor's built-in PID auto-tuner — convenient, but it hides the loop. `GradientTape` is opening that loop and wiring the feedback path myself: I decide exactly when to sample the error (forward pass + loss), how to compute the correction direction (gradients), and when to apply the correction (the optimizer step). This is exactly the kind of control I needed in the GAN notebook, where I had to alternate updates between two networks — something a single `model.fit()` call can't do.

**What I cover:**
- 📥 Setting up data and a basic model for MNIST
- 🔄 Writing a bare-bones custom training loop with `GradientTape`
- 🎯 Adding a stateful accuracy metric to the loop
- 📋 Hooking in a custom callback for epoch-end logging
- 🏗️ Building an ordinary Functional API model (hidden layers, output layer, compile, fit, evaluate)
- 🧪 Practice: revisiting every step end to end


## 🧩 Theory

A standard `model.fit()` call hides three steps inside itself: the forward pass, the loss computation, and the gradient update. A custom training loop makes all three explicit:

$$
\hat{y} = f_\theta(x) \qquad \text{(forward pass)}
$$

$$
\mathcal{L} = \text{SparseCCE}(y, \hat{y}) \qquad \text{(loss)}
$$

$$
\theta \leftarrow \theta - \eta \cdot \text{Adam}\big(\nabla_\theta \mathcal{L}\big) \qquad \text{(gradient update)}
$$

`tf.GradientTape()` is what makes the middle step to the gradient possible: it records every operation applied to watched variables during the forward pass, then uses that recorded graph to compute $\nabla_\theta \mathcal{L}$ via reverse-mode automatic differentiation (the same mechanism behind ordinary backpropagation) when I call `tape.gradient(loss, trainable_weights)`.

### 📡 Telecom analogy

| Custom loop piece | Control-loop equivalent |
|---|---|
| `model.fit()` | A vendor's built-in auto-tuning PID controller — convenient, but closed off |
| `tf.GradientTape()` | Opening the loop and instrumenting it myself — I choose exactly what to measure |
| Forward pass + loss | Sampling the error signal (how far off is the current output) |
| `tape.gradient(...)` | Computing the correction direction from the error |
| `optimizer.apply_gradients(...)` | Applying the correction — the actuator step |
| Custom callback | A logging tap I insert into the loop, independent of the control logic itself |

This is exactly why the GAN notebook couldn't just call `model.fit()` — alternating updates between a generator and a discriminator needs the loop opened up, not a black box.


## Part 1 — 📥 Setup & the Basic Custom Training Loop

I load MNIST, normalize pixel values to $[0,1]$, and batch it into a `tf.data.Dataset`. The model itself is a plain Sequential stack (Flatten → Dense(128, relu) → Dense(10)) — the interesting part isn't the model, it's how I train it.


In [ ]:
!pip install tensorflow numpy


In [ ]:
import os
import warnings
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.callbacks import Callback
import numpy as np

# Suppress all Python warnings
warnings.filterwarnings('ignore')

# Set TensorFlow log level to suppress warnings and info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)


In [ ]:
# Step 2: Define the Model

model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(10)
])


I use `SparseCategoricalCrossentropy(from_logits=True)` since the output layer has no activation (raw logits, not probabilities) — `from_logits=True` tells the loss function to apply the softmax internally rather than expecting it to already be applied.


In [ ]:
# Step 3: Define Loss Function and Optimizer

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()


The loop itself: for each batch, `GradientTape` records the forward pass and loss, `tape.gradient` computes $\nabla_\theta \mathcal{L}$, and `optimizer.apply_gradients` applies the Adam update. I print the loss every 200 steps to keep an eye on convergence without flooding the output.


In [ ]:
# Step 4: Implement the Custom Training Loop

epochs = 2
# train_dataset = train_dataset.repeat(epochs)
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)
for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')

    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            logits = model(x_batch_train, training=True)  # Forward pass
            loss_value = loss_fn(y_batch_train, logits)  # Compute loss

        # Compute gradients and update weights
        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        # Logging the loss every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()}')


## Part 2 — 🎯 Adding an Accuracy Metric

I re-run the same setup, then add `tf.keras.metrics.SparseCategoricalAccuracy()` — a **stateful** metric object: `update_state()` accumulates predictions across a whole epoch, `result()` reads the running value, and `reset_state()` clears it for the next epoch. This is different from computing accuracy fresh each batch — it's a running tally, closer to an integrator than a per-sample instantaneous reading.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize the pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

# Create a batched dataset for training
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)


In [ ]:
# Step 2: Define the Model

model = Sequential([
    Flatten(input_shape=(28, 28)),  # Flatten the input to a 1D vector
    Dense(128, activation='relu'),  # First hidden layer with 128 neurons and ReLU activation
    Dense(10)  # Output layer with 10 neurons for the 10 classes (digits 0-9)
])


In [ ]:
# Step 3: Define Loss Function, Optimizer, and Metric

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)  # Loss function for multi-class classification
optimizer = tf.keras.optimizers.Adam()  # Adam optimizer for efficient training
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()  # Metric to track accuracy during training


In [ ]:
# Step 4: Implement the Custom Training Loop with Accuracy

epochs = 5  # Number of epochs for training

for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')

    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            # Forward pass: Compute predictions
            logits = model(x_batch_train, training=True)
            # Compute loss
            loss_value = loss_fn(y_batch_train, logits)

        # Compute gradients
        grads = tape.gradient(loss_value, model.trainable_weights)
        # Apply gradients to update model weights
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        # Update the accuracy metric
        accuracy_metric.update_state(y_batch_train, logits)

        # Log the loss and accuracy every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')

    # Reset the metric at the end of each epoch
    accuracy_metric.reset_state()


## Part 3 — 📋 Custom Callback for Advanced Logging

I set up the same environment and model once more, then define a `Callback` subclass with an `on_epoch_end` hook. Keras callbacks are a logging/control tap that sits alongside the training logic rather than inside it — in a custom loop, I call the hook manually at the point I want it to fire, since there's no `model.fit()` to invoke it automatically.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize the pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

# Create a batched dataset for training
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)


In [ ]:
# Step 2: Define the Model

model = Sequential([
    Flatten(input_shape=(28, 28)),  # Flatten the input to a 1D vector
    Dense(128, activation='relu'),  # First hidden layer with 128 neurons and ReLU activation
    Dense(10)  # Output layer with 10 neurons for the 10 classes (digits 0-9)
])


In [ ]:
# Step 3: Define Loss Function, Optimizer, and Metric

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)  # Loss function for multi-class classification
optimizer = tf.keras.optimizers.Adam()  # Adam optimizer for efficient training
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()  # Metric to track accuracy during training


In [ ]:
from tensorflow.keras.callbacks import Callback

# Step 4: Implement the Custom Callback
class CustomCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(f'End of epoch {epoch + 1}, loss: {logs.get("loss")}, accuracy: {logs.get("accuracy")}')


In [ ]:
# Step 5: Implement the Custom Training Loop with Custom Callback

epochs = 2
custom_callback = CustomCallback()  # Initialize the custom callback

for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')

    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            # Forward pass: Compute predictions
            logits = model(x_batch_train, training=True)
            # Compute loss
            loss_value = loss_fn(y_batch_train, logits)

        # Compute gradients
        grads = tape.gradient(loss_value, model.trainable_weights)
        # Apply gradients to update model weights
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        # Update the accuracy metric
        accuracy_metric.update_state(y_batch_train, logits)

        # Log the loss and accuracy every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy_metric.result().numpy()}')

    # Call the custom callback at the end of each epoch
    custom_callback.on_epoch_end(epoch, logs={'loss': loss_value.numpy(), 'accuracy': accuracy_metric.result().numpy()})

    # Reset the metric at the end of each epoch
    accuracy_metric.reset_state()  # Use reset_state() instead of reset_states()


## Part 4 — 🏗️ Building a Standard Functional API Model

**A pivot worth flagging:** from here, the material moves away from custom training loops entirely — no more `GradientTape`, no manual gradient steps. It's a standard Functional API model trained with the ordinary `model.fit()`/`model.evaluate()` calls. I keep it in the same notebook since that's how the source material is structured, but it's a genuinely separate topic bolted onto the custom-training-loop material above.

I build the same MNIST classifier shape (Flatten → two 64-unit hidden layers → 10-class output) using `Input`/`Dense` wired through the Functional API rather than `Sequential`.


In [ ]:
from tensorflow.keras.layers import Input, Dense

# Define the input layer
input_layer = Input(shape=(28, 28))  # Input layer with shape (28, 28)

# Flatten the 2D images into 1D vectors before Dense layers
flatten = Flatten()(input_layer)

# Define hidden layers
hidden_layer1 = Dense(64, activation='relu')(flatten)  # First hidden layer with 64 neurons and ReLU activation
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)  # Second hidden layer with 64 neurons and ReLU activation


`Flatten()` converts each 28×28 image into a 784-element 1D vector so it can feed into `Dense` layers, and each hidden layer takes the previous layer's output as its own input — the Functional API's signature "call it like a function" wiring style.


**Another inconsistency worth flagging:** the original instructions describe this as "a binary classification problem" with "one unit with a sigmoid activation function," but the actual code below builds a 10-unit `softmax` output — correct for 10-class MNIST digit classification, but not what the prose describes. I build exactly what the code specifies (10-way softmax), since that's what's actually needed for this dataset.


In [ ]:
output_layer = Dense(10, activation='softmax')(hidden_layer2)


`Dense(10, activation='softmax')` gives one output neuron per digit class (0–9), with `softmax` ensuring all 10 outputs sum to 1 — interpretable as class probabilities, unlike the raw logits I used in Parts 1–3.


In [ ]:
model = Model(inputs=input_layer, outputs=output_layer)


`Model(inputs=input_layer, outputs=output_layer)` wires the input straight through both hidden layers to the output, creating a trainable Keras model from the graph I just built.


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # Correct for integer labels 0-9
    metrics=['accuracy']
)


Unlike Parts 1–3 where I used `from_logits=True` (since the model output raw logits), here the output layer already applies `softmax`, so the loss string `'sparse_categorical_crossentropy'` doesn't need the logits flag — it expects probabilities, which is exactly what it's getting.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import numpy as np

# Load and preprocess MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0  # Normalize pixel values to [0, 1]

# Train the model
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=32
)


No `GradientTape` here — `model.fit()` handles the forward pass, loss, and gradient update internally. This is the "closed loop" I referenced in the Theory section: the exact same underlying mechanics as Parts 1-3, just packaged behind a single call.


In [ ]:
# Example test data (in practice, use real dataset)
loss, accuracy = model.evaluate(x_test, y_test)

print(f'Test loss:     {loss:.4f}')
print(f'Test accuracy: {accuracy:.4f}')


**One more small inconsistency:** the comment above says "in practice, use real dataset," but `x_test`/`y_test` here *are* the real MNIST test set, reloaded a couple of cells earlier — leftover boilerplate text from wherever this snippet was originally copied from. `model.evaluate()` computes loss and accuracy on that held-out data, giving me a read on generalization rather than just training performance.


## 🧪 Practice: Revisiting the Pipeline

The source material repeats the same nine steps as a set of practice exercises — each with a blank "write your code here" cell followed by a hidden solution. I fill in every one below. Most repeat the exact same pattern already covered above; where a practice version differs in some small but real way, I call it out rather than presenting it as if it were identical.

### Practice 1 — Basic custom training loop

Same pattern as Part 1, just 5 epochs instead of 2 and per-epoch (not per-200-step) logging.


In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)

# Step 2: Define the Model
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(10)
])

# Step 3: Define Loss Function and Optimizer
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()

# Step 4: Implement the Custom Training Loop
for epoch in range(5):
    for x_batch, y_batch in train_dataset:
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss = loss_fn(y_batch, logits)
        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
    print(f'Epoch {epoch + 1}: Loss = {loss.numpy()}')


### Practice 2 — Adding an accuracy metric

**Difference worth noting:** this solution's setup only keeps `x_train`/`y_train` (via `_` for the test split) and drops the `x_test`/`y_test` I'd normally hang on to — fine here since this exercise only trains, but a reminder to double-check what a copy-pasted setup snippet actually keeps.


In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# Step 1: Set Up the Environment
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train / 255.0
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)

# Step 2: Define the Model
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(10)
])

# Step 3: Define Loss Function, Optimizer, and Metric
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()

# Step 4: Implement the Custom Training Loop with Accuracy Tracking
epochs = 5
for epoch in range(epochs):
    for x_batch, y_batch in train_dataset:
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss = loss_fn(y_batch, logits)
        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        accuracy_metric.update_state(y_batch, logits)
    print(f'Epoch {epoch + 1}: Loss = {loss.numpy()} Accuracy = {accuracy_metric.result().numpy()}')
    accuracy_metric.reset_state()


### Practice 3 — Custom callback for advanced logging

**Difference worth noting:** this solution defines the model with `tf.keras.Input(shape=(28, 28))` as an explicit first layer inside the `Sequential([...])` list, rather than passing `input_shape=` to the first `Flatten` layer like every earlier version did — both work, but the explicit `Input` layer is the more modern, recommended Keras style.


In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.callbacks import Callback

# Step 1: Set Up the Environment
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train / 255.0
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32)

# Step 2: Define the Model
model = Sequential([
    tf.keras.Input(shape=(28, 28)),  # Updated Input layer syntax
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10)
])

# Step 3: Define Loss Function, Optimizer, and Metric
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()
accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy()

# Step 4: Implement the Custom Callback
class CustomCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f'End of epoch {epoch + 1}, loss: {logs.get("loss")}, accuracy: {logs.get("accuracy")}')

# Step 5: Implement the Custom Training Loop with Custom Callback
custom_callback = CustomCallback()

for epoch in range(5):
    for x_batch, y_batch in train_dataset:
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss = loss_fn(y_batch, logits)
        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        accuracy_metric.update_state(y_batch, logits)
    custom_callback.on_epoch_end(epoch, logs={'loss': loss.numpy(), 'accuracy': accuracy_metric.result().numpy()})
    accuracy_metric.reset_state()  # Updated method


### Practice 5 — Hidden layers and output layer

(The source material skips a "Practice 4" — I keep its numbering as-is.) Same pattern as Part 4's hidden-layer/output-layer construction, rebuilt cleanly for MNIST's actual shape (28×28 input, 10 digit classes) — and again, softmax over 10 classes rather than the "sigmoid" the instructions mention.


In [ ]:
from tensorflow.keras.layers import Input, Dense, Flatten

# Re-define layers for MNIST (28x28 images, 10 digit classes)
input_layer   = Input(shape=(28, 28))                          # Input: 28x28 grayscale images
flatten       = Flatten()(input_layer)                         # Flatten to 784-dim vector
hidden_layer1 = Dense(64, activation='relu')(flatten)          # First hidden layer
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)    # Second hidden layer
output_layer  = Dense(10, activation='softmax')(hidden_layer2) # Output: 10 classes (digits 0-9)


### Practice 6 — Create the model


In [ ]:
# Create the model by specifying input and output layers
model = Model(inputs=input_layer, outputs=output_layer)


### Practice 7 — Compile the model


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # Correct loss for integer labels (0-9)
    metrics=['accuracy']
)


### Practice 8 — Train the model


In [ ]:
history = model.fit(
    x_train, y_train,        # Training data and labels
    epochs=5,                # Number of full passes over the training data
    batch_size=32,           # Number of samples per gradient update
)

print('Training complete.')


### Practice 9 — Evaluate the model


In [ ]:
# Evaluate the trained model on the held-out test dataset
test_loss, test_accuracy = model.evaluate(x_test, y_test)

# Print the evaluation results
print(f'Test loss:     {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')


## 📊 Summary

| Concept | What I did | Why it matters |
|---|---|---|
| 📥 Setup | Loaded MNIST, normalized to $[0,1]$, batched via `tf.data.Dataset` | Standard prep for any Keras training loop |
| 🔄 Basic custom loop | `GradientTape` → forward pass → loss → gradients → `apply_gradients` | Makes every step of training explicit instead of hidden inside `.fit()` |
| 🎯 Accuracy metric | Added a stateful `SparseCategoricalAccuracy` — `update_state`/`result`/`reset_state` | Running per-epoch accuracy, not per-batch |
| 📋 Custom callback | Subclassed `Callback`, called `on_epoch_end` manually | Logging hook independent of the training logic itself |
| 🏗️ Functional API pivot | Built a standard Input→Dense→Dense→Dense model, trained via `model.fit()` | Same underlying mechanics as the custom loop, packaged behind one call |
| 🧪 Practice | Re-ran every step, filled in every blank | Reinforced the pattern, surfaced small but real differences between "versions" |

**Where custom loops actually pay off:** anywhere the training dynamic doesn't fit `model.fit()`'s single-model, single-loss assumption — like the GAN notebook's alternating generator/discriminator updates. `GradientTape` is the tool that makes that kind of non-standard training loop possible.


## 🧪 Sandbox

Space to keep experimenting beyond the practice exercises:

- Add a validation pass inside the custom loop (a second `GradientTape`-free forward pass over a held-out batch) and log both train/val loss per epoch
- Wire a real Keras `Callback` (like `EarlyStopping`) into a custom loop manually, calling `on_epoch_end`/`on_train_end` at the right points
- Try `tf.function`-decorating the training step to see the speed difference from graph compilation
- Rebuild the GAN training loop from the earlier notebook using the same `GradientTape` pattern shown here, side by side with the `train_on_batch` version
- Track more than one metric (e.g. precision/recall) in the same custom loop using multiple `tf.keras.metrics` objects


In [ ]:
# 🧪 Sandbox — experiment here
